In [ ]:
!pip install -U bitsandbytes accelerate zeus-ml einops -q
!pip install transformers==4.50.3 -q

In [ ]:
from kaggle_secrets import UserSecretsClient
import os
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("HF token loaded:", os.environ["HF_TOKEN"][:8], "...")

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Visible devices:", torch.cuda.device_count())   # must be 1
print("Device:", torch.cuda.get_device_name(0))        # Tesla T4
print("transformers:", __import__("transformers").__version__)  # must be 4.50.3

In [ ]:
from zeus.monitor import ZeusMonitor
monitor = ZeusMonitor(gpu_indices=[0])
print("Idle power (W):", monitor.gpus.getInstantPowerUsage(0) / 1000)

In [ ]:
from huggingface_hub import HfApi

MODEL_ID = "allenai/Molmo-7B-D-0924"
REVISION = HfApi().model_info(MODEL_ID).sha
print("Pinned revision:", REVISION)
print("transformers pin: 4.50.3 (matches upstream PR #43, merged compatibility fix)")
print("No source patching applied — running Molmo's code as shipped at this revision.")

In [ ]:
from transformers import AutoModelForCausalLM, AutoProcessor, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID, revision=REVISION, trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=REVISION, trust_remote_code=True,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map={"": 0},
)
model.eval()

print("Loaded on:", next(model.parameters()).device)
print("GPU memory (GB):", torch.cuda.memory_allocated(0) / 1e9)

In [ ]:
from transformers import GenerationConfig

MAX_NEW_TOKENS = 32
VQA_PROMPT_SUFFIX = "Answer the question using a single word or phrase."

def molmo_infer(image, question):
    """Returns (predicted_text, tokens_generated). Molmo has no chat template."""
    inputs = processor.process(
        images=[image],
        text=f"{question} {VQA_PROMPT_SUFFIX}",
    )
    # processor.process returns UNBATCHED tensors -> unsqueeze to batch of 1
    inputs = {k: v.to(model.device).unsqueeze(0) for k, v in inputs.items()}
    # cast every float tensor to fp16 to match compute dtype
    # (model card casts only "images"; this is strictly safer)
    for k, v in inputs.items():
        if v.dtype.is_floating_point:
            inputs[k] = v.to(torch.float16)

    with torch.no_grad():
        output = model.generate_from_batch(
            inputs,
            GenerationConfig(
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,                  # §4.6 greedy, non-negotiable
                stop_strings="<|endoftext|>",
            ),
            tokenizer=processor.tokenizer,
        )

    gen = output[0, inputs["input_ids"].size(1):]
    text = processor.tokenizer.decode(gen, skip_special_tokens=True).strip()
    return text, int(gen.shape[0]), int(inputs["input_ids"].size(1))

In [ ]:
import json, time
from PIL import Image

DATA_DIR = "/kaggle/input/notebooks/mmrafi14/ecovlm-data-prep/vqav2_subset"
with open(f"{DATA_DIR}/metadata.json") as f:
    _S = json.load(f)

for s in _S[:3]:
    img = Image.open(f"{DATA_DIR}/images/{s['image_file']}")
    if img.mode != "RGB":
        img = img.convert("RGB")
    t0 = time.perf_counter()
    pred, ntok, prefill = molmo_infer(img, s["question"])
    dt = time.perf_counter() - t0
    print(f"Q: {s['question']}")
    print(f"  pred: {pred!r}  | new_tokens={ntok} | prefill={prefill} | {dt:.2f}s")
    print(f"  gt  : {s['multiple_choice_answer']}\n")

In [ ]:
import os, json, csv, time
from datetime import datetime, timezone     # timezone REQUIRED

MODEL_NAME = "molmo-7b-d"
DATA_DIR = "/kaggle/input/notebooks/mmrafi14/ecovlm-data-prep/vqav2_subset"
OUT_DIR = "/kaggle/working"
LOG_FILE = os.path.join(OUT_DIR, f"{MODEL_NAME}_telemetry.csv")
MANIFEST_FILE = os.path.join(OUT_DIR, f"{MODEL_NAME}_manifest.json")

N_RUNS = 5
MAX_NEW_TOKENS = 32
VQA_PROMPT_SUFFIX = "Answer the question using a single word or phrase."

with open(os.path.join(DATA_DIR, "metadata.json")) as f:
    SAMPLES = json.load(f)

print(f"Samples loaded: {len(SAMPLES)}")
print(f"Log file: {LOG_FILE}")

In [ ]:
def vqa_accuracy(predicted: str, ground_truth_answers: list) -> float:
    import re
    def normalise(s):
        s = s.lower().strip()
        s = re.sub(r"[^\w\s]", "", s)
        for article in ["a ", "an ", "the "]:
            if s.startswith(article):
                s = s[len(article):]
        return s.strip()
    pred_norm = normalise(predicted)
    matches = sum(1 for a in ground_truth_answers if normalise(a) == pred_norm)
    return min(matches / 3, 1.0)

In [ ]:
def load_manifest():
    if os.path.exists(MANIFEST_FILE):
        with open(MANIFEST_FILE) as f:
            return json.load(f)
    return {"completed_runs": [], "current_run": None}

def save_manifest(manifest):
    with open(MANIFEST_FILE, "w") as f:
        json.dump(manifest, f, indent=2)

manifest = load_manifest()
print("Completed runs so far:", manifest["completed_runs"])

In [ ]:
print("Warm-up: 5 discarded inferences...")
_img_dir = os.path.join(DATA_DIR, "images")
for s in SAMPLES[:5]:
    _im = Image.open(os.path.join(_img_dir, s["image_file"]))
    if _im.mode != "RGB":
        _im = _im.convert("RGB")
    _ = molmo_infer(_im, s["question"])
print("Warm-up complete — results discarded.")

In [ ]:
print("Measuring idle baseline (30 seconds)...")
idle_readings = []
for _ in range(300):
    idle_readings.append(monitor.gpus.getInstantPowerUsage(0) / 1000)
    time.sleep(0.1)

IDLE_POWER_W = sum(idle_readings) / len(idle_readings)
print(f"Idle baseline: {IDLE_POWER_W:.3f} W")
print(f"Idle std dev:  {(sum((x-IDLE_POWER_W)**2 for x in idle_readings)/len(idle_readings))**0.5:.3f} W")

In [ ]:
import random
random.seed(42)
dry_samples_random = random.sample(SAMPLES, 20)
print("Sampled sample_idx:", sorted(s["sample_idx"] for s in dry_samples_random))

In [ ]:
IMG_DIR = os.path.join(DATA_DIR, "images")

write_header = not os.path.exists(LOG_FILE)
log_f = open(LOG_FILE, "a", newline="", buffering=1)   # line-buffered = crash-safe
writer = csv.DictWriter(log_f, fieldnames=[
    "run_id", "sample_idx", "question_id", "image_id",
    "question", "predicted", "vqa_accuracy",
    "energy_gross_j", "energy_net_j", "avg_power_w",
    "idle_power_w", "elapsed_s", "tokens_generated",
    "prefill_tokens", "timestamp"])   # prefill_tokens added — Molmo-specific
if write_header:
    writer.writeheader()

for run_id in range(1, N_RUNS + 1):
    if run_id in manifest["completed_runs"]:
        print(f"Run {run_id} already complete — skipping.")
        continue

    print(f"\n{'='*50}\nStarting run {run_id} of {N_RUNS}\n{'='*50}")
    manifest["current_run"] = run_id
    save_manifest(manifest)

    # resume mid-run: skip samples already logged for this run
    done_idx = set()
    if os.path.exists(LOG_FILE):
        with open(LOG_FILE) as _f:
            for row in csv.DictReader(_f):
                if int(row["run_id"]) == run_id:
                    done_idx.add(int(row["sample_idx"]))
    if done_idx:
        print(f"Resuming run {run_id}: {len(done_idx)} samples already logged, skipping.")

    for s in SAMPLES:
        idx = s["sample_idx"]
        if idx in done_idx:
            continue

        image = Image.open(os.path.join(IMG_DIR, s["image_file"]))
        if image.mode != "RGB":
            image = image.convert("RGB")

        window_name = f"run{run_id}_sample{idx}"
        monitor.begin_window(window_name)
        t0 = time.perf_counter()
        predicted, tokens_generated, prefill = molmo_infer(image, s["question"])
        elapsed = time.perf_counter() - t0
        measurement = monitor.end_window(window_name)

        energy_gross = measurement.total_energy
        avg_power    = energy_gross / elapsed if elapsed > 0 else 0
        energy_net   = max((avg_power - IDLE_POWER_W) * elapsed, 0)
        acc          = vqa_accuracy(predicted, s["answers"])

        writer.writerow({
            "run_id": run_id, "sample_idx": idx,
            "question_id": s["question_id"], "image_id": s["image_id"],
            "question": s["question"], "predicted": predicted,
            "vqa_accuracy": round(acc, 4),
            "energy_gross_j": round(energy_gross, 4),
            "energy_net_j": round(energy_net, 4),
            "avg_power_w": round(avg_power, 3),
            "idle_power_w": round(IDLE_POWER_W, 3),
            "elapsed_s": round(elapsed, 4),
            "tokens_generated": tokens_generated,
            "prefill_tokens": prefill,
            "timestamp": datetime.now(timezone.utc).isoformat()})

    manifest["completed_runs"].append(run_id)
    manifest["current_run"] = None
    save_manifest(manifest)
    print(f"Run {run_id} complete.")

log_f.close()
print("\nAll runs complete.")
print(f"Total rows logged: {sum(1 for _ in open(LOG_FILE)) - 1}")

In [ ]:
import shutil
backup = LOG_FILE.replace(".csv", "_backup.csv")
shutil.copy(LOG_FILE, backup)
print("Rows in log:", sum(1 for _ in open(LOG_FILE)) - 1)
print("Backup written:", backup, os.path.exists(backup))